# 23 — Synthetic Feature Construction, Stacking, & Index Resets
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive guide to series construction, horizontal vs vertical concatenation, index repetition traps, and reset_index vs reindex distinctions in Pandas.*

---

## 📌 Executive Summary & Interview Expectations
In interview coding challenges, candidates are often asked to generate synthetic datasets from random distributions, combine them across axes, and correct structural index defects:
1. **Series Assembly (`axis=1` vs `axis=0`)**: Combining independent Series horizontally into tabular records vs vertically into unified streams.
2. **The "Index Going Only Until 99" Trap**: Why stacking Series of length 100 results in repeated index labels ($0..99$ three times) and why `len(df)` is 300 while `df.index.max()` is only 99.
3. **`reset_index()` vs `reindex()`**: One of the most frequently asked Python/Pandas technical interview distinctions.
4. **Data Modeling & Feature Engineering**: Building calculated ratios (e.g. bathroom-to-bedroom ratios, luxury segmentation) idiomatically.

## 1. Environment Setup & Synthetic Data Generation

We generate 3 Series representing housing market attributes (length = 100 each):
- `bedrs`: Random integers $[1, 4]$
- `bathrs`: Random integers $[1, 3]$
- `price_sqr_meter`: Random integers $[10,000, 30,000]$

In [1]:
import numpy as np
import pandas as pd

# Set seed for reproducible results
np.random.seed(42)

data1 = pd.Series(np.random.randint(1, 5, size=100), name="bedrs")
data2 = pd.Series(np.random.randint(1, 4, size=100), name="bathrs")
data3 = pd.Series(np.random.randint(10000, 30001, size=100), name="price_sqr_meter")

print("Generated 3 Series of length:", len(data1))
print("Sample bedrs:", data1.head(3).tolist())
print("Sample bathrs:", data2.head(3).tolist())
print("Sample price_sqr_meter:", data3.head(3).tolist())

Generated 3 Series of length: 100
Sample bedrs: [3, 4, 1]
Sample bathrs: [3, 2, 2]
Sample price_sqr_meter: [23931, 13627, 26157]


## 2. Horizontal DataFrame Assembly (`axis=1`)

### 💡 Interview Tip: Naming Columns During Construction
Instead of creating default numeric column headers (`0, 1, 2`) and calling `.rename(columns=...)`, either:
1. Pass named Series directly to `pd.concat(..., axis=1)`, or
2. Construct using a dictionary: `pd.DataFrame({'bedrs': data1, 'bathrs': data2, 'price_sqr_meter': data3})`.

In [2]:
# Method 1: Concat named Series along columns
df = pd.concat([data1, data2, data3], axis=1)

print("Constructed Housing DataFrame (Shape:", df.shape, "):")
df.head(4)

Constructed Housing DataFrame (Shape: (100, 3) ):


,bedrs,bathrs,price_sqr_meter
0,3,3,23931
1,4,2,13627
2,1,2,26157
3,3,2,20173


## 3. Vertical Stacking (`axis=0`) & The Repetition Trap

### 🚨 Top Interview Gotcha: "It's only going until 99!"
When stacking the 3 Series vertically:
`bigcolumn = pd.concat([data1, data2, data3], axis=0)`
- The total row count is $100 + 100 + 100 = 300$ rows.
- However, each original Series possessed index labels $0..99$.
- The resulting Series has **triplicate index labels**: 0, 1, ..., 99, 0, 1, ..., 99, 0, 1, ..., 99!
- Looking at `bigcolumn.loc[0]` returns **3 values**, not 1!

In [3]:
# Stacking vertically
bigcolumn = pd.concat([data1, data2, data3], axis=0)

print(f"Shape of bigcolumn: {bigcolumn.shape}")
print(f"Index is unique: {bigcolumn.index.is_unique}")
print(f"Max index label: {bigcolumn.index.max()}")
print(f"Values at index label 0:\n{bigcolumn.loc[0]}")

Shape of bigcolumn: (300,)
Index is unique: False
Max index label: 99
Values at index label 0:
0        3
0        3
0    23931
dtype: int64


## 4. Restoring Index Contiguity: `reset_index`

### 💡 `reset_index(drop=True)` vs `reindex()`
- `reset_index(drop=True)`: Replaces the existing index with a clean, contiguous `RangeIndex(0, len(df))`.
- `reindex(new_labels)`: Conforms the data to match a specific set of target labels, inserting `NaN` where labels are missing.

In [4]:
# Reset index immutably
bigcolumn_clean = bigcolumn.reset_index(drop=True)

print(f"Cleaned bigcolumn shape: {bigcolumn_clean.shape}")
print(f"Index is strictly unique: {bigcolumn_clean.index.is_unique}")
print(f"Index range: [{bigcolumn_clean.index.min()} ... {bigcolumn_clean.index.max()}]")
print(f"Single scalar at loc[0]: {bigcolumn_clean.loc[0]}")

Cleaned bigcolumn shape: (300,)
Index is strictly unique: True
Index range: [0 ... 299]
Single scalar at loc[0]: 3


---
## 🎯 5. Technical Interview Corner: Tricky Questions & Drills

### Q1: What is the fundamental difference between `df.reset_index()` and `df.reindex()`?
**Answer**:
1. **`df.reset_index(drop=...)`**:
   - Resets the index to the default integer `RangeIndex` ($0, 1, 2, \dots, N-1$).
   - If `drop=False` (default), the old index is pushed into a new column.
   - If `drop=True`, the old index is discarded.
2. **`df.reindex(labels=...)`**:
   - Does NOT generate a default range. It aligns the existing DataFrame to a **specified collection of labels**.
   - If a specified label is NOT present in the original DataFrame, Pandas inserts `NaN` (or `fill_value`).
   - If an existing row's label is NOT in `new_labels`, that row is **dropped**.

---

### Q2: Advanced Interview Coding Challenge: Real Estate Market Segmentation
**Challenge**:
Using the `df` housing dataset:
1. Compute the **Bathroom-to-Bedroom Ratio** (`bath_per_bed = bathrs / bedrs`).
2. Classify properties into **Price Tiers**:
   - `'Budget'`: `price_sqr_meter < 15,000`
   - `'Standard'`: `15,000 <= price_sqr_meter < 25,000`
   - `'Luxury'`: `price_sqr_meter >= 25,000`
3. Find the **average bathroom-to-bedroom ratio** and **total inventory** per Price Tier!

In [5]:
# Interview Solution: Feature Engineering & Tier Analysis
df_market = df.assign(
    bath_to_bed_ratio=lambda d: (d["bathrs"] / d["bedrs"]).round(2),
    price_tier=lambda d: pd.cut(
        d["price_sqr_meter"],
        bins=[-np.inf, 15000, 25000, np.inf],
        labels=["Budget", "Standard", "Luxury"]
    )
)

market_summary = df_market.groupby("price_tier", observed=False).agg(
    total_listings=("bedrs", "count"),
    avg_price_sqm=("price_sqr_meter", "mean"),
    avg_bath_bed_ratio=("bath_to_bed_ratio", "mean")
).round(2)

print("Housing Market Tier Analysis:")
display(market_summary)

Housing Market Tier Analysis:


,total_listings,avg_price_sqm,avg_bath_bed_ratio
price_tier,,,
Budget,25,12476.64,1.11
Standard,51,20094.20,0.94
Luxury,24,27773.54,0.99
